# 2004. The Number of Seniors and Juniors to Join the Company

## Description
We have a table called **Candidates**:

| Column Name | Type |
|-------------|------|
| employee_id | int  |
| experience  | enum ('Senior', 'Junior') |
| salary      | int  |

- `employee_id` is the primary key.  
- `experience` indicates whether the candidate is a **Senior** or **Junior**.  
- `salary` is the monthly salary of the candidate.  

---

## Hiring Criteria
- The company has a **budget of $70,000**.  
- First, hire the **maximum number of Seniors**.  
- With the remaining budget, hire the **maximum number of Juniors**.  

The task is to write an SQL query to find the number of Seniors and Juniors hired under these criteria.

---

## Example 1

**Input:**

```sql
CREATE TABLE Candidates (
    employee_id INT PRIMARY KEY,
    experience ENUM('Senior','Junior'),
    salary INT
);

INSERT INTO Candidates (employee_id, experience, salary) VALUES
(1, 'Junior', 10000),
(9, 'Junior', 10000),
(2, 'Senior', 20000),
(11, 'Senior', 20000),
(13, 'Senior', 50000),
(4, 'Junior', 40000);


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("experience", StringType(), True),  # use String instead of ENUM
    StructField("salary", IntegerType(), True)
])
data = [
    (1, "Junior", 10000),
    (9, "Junior", 10000),
    (2, "Senior", 20000),
    (11, "Senior", 20000),
    (13, "Senior", 50000),
    (4, "Junior", 40000)
]

df = spark.createDataFrame(data, schema)
df.createOrReplaceTempView("Candidates")


In [0]:
%sql
select *  from Candidates order by experience desc  , salary asc 

In [0]:
%sql
with cte_s as (
  Select employee_id , experience , salary , sum(salary)over(order by salary asc ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) as purse  from Candidates where experience = 'Senior'
)
, cte_j
(
  Select employee_id , experience , salary , coalesce ((select max(purse) from cte_s where purse <= 70000 ),0)
  +
  sum(salary)over(order by salary asc ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) as purse from 
  Candidates where experience = 'Junior'
)
Select 'Senior' as experience  , count(employee_id) accepted_candidates from cte_s where purse <= 70000
union all
Select 'Junior' as experience , count(employee_id) accepted_candidates  from cte_j where purse <= 70000


In [0]:
%sql
drop table if exists Candidates

In [0]:


# case 2
data_2 = [
    (1, "Junior", 10000),
    (9, "Junior", 10000),
    (2, "Senior", 80000),
    (11, "Senior", 80000),
    (13, "Senior", 80000),
    (4, "Junior", 40000)
]

# Create DataFrame
df_2 = spark.createDataFrame(data_2, schema)


# Register as temp view for SQL queries
df_2.createOrReplaceTempView("Candidates")

# Quick check
df.show()



# Learning Notes: Rolling Sum in SQL

## Key Insight
I learned how to calculate a **rolling sum** using SQL window functions. Initially, I tried using `LAG` to reference previous values, but I realized that it cannot be used to compute a cumulative sum beyond a certain point. That approach was insufficient.

---

## My Thought Process
- At first, I avoided separating candidates into **Junior** and **Senior** categories.  
- Later, I realized that this segregation was necessary to correctly apply the hiring logic.

---

## Approach Taken
1. **Filter Seniors**  
   - Created a column for cumulative/rolling sum of salaries.  
   - In the second CTE, selected the maximum rolling sum that was still less than the budget (`70000`).  

2. **Handle Juniors**  
   - Added the cumulative sum of juniors to the remaining budget.  
   - Used `COALESCE` to handle cases where no seniors were hired. (`ISNULL` could also be used, but I prefer `COALESCE`.)

3. **Combine Results**  
   - Used `UNION ALL` to merge results for seniors and juniors.  
   - Avoided plain `UNION` to prevent accidental removal of duplicates.

---

## Learnings
- `LAG` is useful for referencing previous rows, but not for cumulative sums.  
- Segregating data by category (Senior vs Junior) is essential for clarity.  
- `COALESCE` is a reliable way to handle nulls when no rows are returned.  
- `UNION ALL` is safer than `UNION` when duplicates are possible and intentional.
